## Example 1: Basic Contract Decorators and Preconditions

In [ ]:
import dpcontracts
from dpcontracts import require, ensure, PreconditionError, PostconditionError
import math

print("Example 1: Basic Contract Decorators and Preconditions")

try:
    print("✓ Basic contract decorators and preconditions:")
    
    # 1. Package information
    print("  1. Package information:")
    print(f"    dpcontracts module imported successfully")
    print(f"    Package location: {dpcontracts.__file__}")
    
    contract_items = [item for item in dir(dpcontracts) if not item.startswith('_')]
    print(f"    Available components: {len(contract_items)} items")
    
    # 2. Basic precondition examples using correct dpcontracts syntax
    print("  2. Basic precondition examples:")
    
    @require("x must be positive", lambda args: args.x > 0)
    def square_root(x):
        """Calculate square root with precondition."""
        return math.sqrt(x)
    
    @require("n must be non-negative", lambda args: args.n >= 0) 
    @require("n must be integer", lambda args: isinstance(args.n, int))
    def factorial(n):
        """Calculate factorial with preconditions."""
        if n <= 1:
            return 1
        return n * factorial(n - 1)
    
    @require("divisor cannot be zero", lambda args: args.divisor != 0)
    def safe_divide(dividend, divisor):
        """Perform division with zero-division check."""
        return dividend / divisor
    
    # Test successful cases
    print("    Testing successful cases:")
    
    try:
        result = square_root(16)
        print(f"      square_root(16) = {result}")
    except Exception as e:
        print(f"      square_root(16) failed: {type(e).__name__}")
        raise
    
    try:
        result = factorial(5)
        print(f"      factorial(5) = {result}")
    except Exception as e:
        print(f"      factorial(5) failed: {type(e).__name__}")
        raise
    
    try:
        result = safe_divide(10, 2)
        print(f"      safe_divide(10, 2) = {result}")
    except Exception as e:
        print(f"      safe_divide(10, 2) failed: {type(e).__name__}")
        raise
    
    # Test precondition failures
    print("    Testing precondition failures:")
    
    try:
        result = square_root(-4)
    except PreconditionError as e:
        assert str(e) == "x must be positive", str(e)
        print(f"      square_root(-4) correctly failed: PreconditionError")
    else:
        raise AssertionError(f"square_root(-4) = {result}: expected PreconditionError('x must be positive')")
    
    print("✓ Basic contract decorators completed")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dpcontracts example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Postconditions and Result Validation

In [ ]:
from dpcontracts import require, ensure, PreconditionError, PostconditionError

print("Example 2: Postconditions and Result Validation")

try:
    print("✓ Postconditions and result validation:")
    
    # 1. Basic postcondition examples
    print("  1. Basic postcondition examples:")
    
    @require("x must be non-negative", lambda args: args.x >= 0)
    @ensure("result must be non-negative", lambda args, result: result >= 0)
    def square_root_safe(x):
        """Calculate square root with pre and post conditions."""
        import math
        return math.sqrt(x)
    
    @require("list cannot be empty", lambda args: len(args.numbers) > 0)
    @ensure("average must be within range", lambda args, result: min(args.numbers) <= result <= max(args.numbers))
    def calculate_average(numbers):
        """Calculate average with result validation."""
        return sum(numbers) / len(numbers)
    
    @require("text must be non-empty string", lambda args: args.text and isinstance(args.text, str))
    @ensure("result cannot be longer than input", lambda args, result: len(result) <= len(args.text))
    @ensure("result must be string", lambda args, result: isinstance(result, str))
    def remove_vowels(text):
        """Remove vowels with result validation."""
        vowels = "aeiouAEIOU"
        return ''.join(char for char in text if char not in vowels)
    
    # Test postcondition examples
    print("    Testing postcondition examples:")
    
    test_cases = [
        ("square_root_safe(25)", lambda: square_root_safe(25)),
        ("calculate_average([1, 2, 3, 4, 5])", lambda: calculate_average([1, 2, 3, 4, 5])),
        ("remove_vowels('hello world')", lambda: remove_vowels('hello world'))
    ]
    
    for test_name, test_func in test_cases:
        try:
            result = test_func()
            print(f"      {test_name} = {result}")
        except Exception as e:
            print(f"      {test_name} failed: {type(e).__name__}")
            raise
    
    # 2. Simple postcondition violation test
    print("  2. Postcondition violation test:")
    
    @ensure("result length must equal input length", lambda args, result: len(result) == len(args.items))
    def broken_function(items):
        """Function that violates its postcondition."""
        return items[:-1]  # Remove last item, violating length postcondition
    
    try:
        result = broken_function([1, 2, 3])
    except PostconditionError as e:
        assert str(e) == "result length must equal input length", str(e)
        print(f"      broken_function correctly failed: PostconditionError")
    else:
        raise AssertionError(f"broken_function([1, 2, 3]) = {result}: expected PostconditionError")
    
    print("✓ Postconditions and result validation completed")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dpcontracts example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Class Invariants and Object-Oriented Contracts

In [ ]:
from dpcontracts import require, ensure, invariant

print("Example 3: Class Invariants and Object-Oriented Contracts")

try:
    print("✓ Class invariants and object-oriented contracts:")
    
    # 1. Simple class with invariants
    print("  1. Simple class with invariants:")
    
    @invariant("balance must never be negative", lambda self: self.balance >= 0)
    class SimpleBankAccount:
        """Bank account with balance invariant."""
        
        @require("initial balance must be non-negative", lambda args: args.initial_balance >= 0)
        def __init__(self, initial_balance=0):
            self.balance = initial_balance
        
        @require("deposit amount must be positive", lambda args: args.amount > 0)
        @ensure("returns new balance", lambda args, result: result == args.self.balance)
        def deposit(self, amount):
            """Deposit money with validation."""
            self.balance += amount
            return self.balance
        
        @require("withdrawal amount must be positive", lambda args: args.amount > 0)
        @require("insufficient funds", lambda args: args.amount <= args.self.balance)
        def withdraw(self, amount):
            """Withdraw money with validation."""
            self.balance -= amount
            return self.balance
    
    print("    Testing bank account with invariants:")
    
    try:
        account = SimpleBankAccount(100)
        print(f"      Created account with balance: ${account.balance}")
        
        account.deposit(50)
        print(f"      After deposit: ${account.balance}")
        
        account.withdraw(30)
        print(f"      After withdrawal: ${account.balance}")
        
    except Exception as e:
        print(f"      Bank account test failed: {type(e).__name__}")
        raise
    
    # 2. Container class with capacity constraints
    print("  2. Container class with capacity:")
    
    @invariant("cannot exceed capacity", lambda self: len(self.items) <= self.capacity)
    @invariant("capacity must be positive", lambda self: self.capacity > 0)
    class SimpleContainer:
        """Container with capacity constraints."""
        
        @require("capacity must be positive", lambda args: args.capacity > 0)
        def __init__(self, capacity):
            self.capacity = capacity
            self.items = []
        
        @require("container is full", lambda args: len(args.self.items) < args.self.capacity)
        def add_item(self, item):
            """Add item to container."""
            self.items.append(item)
            return len(self.items)
        
        @require("container is empty", lambda args: len(args.self.items) > 0)
        def remove_item(self):
            """Remove last item from container."""
            return self.items.pop()
        
        def is_full(self):
            """Check if container is at capacity."""
            return len(self.items) >= self.capacity
    
    print("    Testing container with invariants:")
    
    try:
        container = SimpleContainer(3)
        print(f"      Created container with capacity: {container.capacity}")
        
        container.add_item("item1")
        container.add_item("item2")
        print(f"      Added items: {container.items}")
        print(f"      Is full: {container.is_full()}")
        
        removed = container.remove_item()
        print(f"      Removed item: {removed}")
        print(f"      Remaining items: {container.items}")
        
    except Exception as e:
        print(f"      Container test failed: {type(e).__name__}")
        raise
    
    # 3. Benefits of class invariants
    print("  3. Benefits of class invariants:")
    
    benefits = [
        "Maintain object consistency throughout lifetime",
        "Catch state corruption bugs early",
        "Document class behavior and constraints",
        "Provide automatic validation"
    ]
    
    for benefit in benefits:
        print(f"      • {benefit}")
    
    print("✓ Class invariants and object-oriented contracts completed")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dpcontracts example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Advanced Contract Patterns and Debugging

In [ ]:
from dpcontracts import require, ensure, PreconditionError
import math

print("Example 4: Advanced Contract Patterns and Debugging")

try:
    print("✓ Advanced contract patterns and debugging:")
    
    # 1. Complex contract expressions
    print("  1. Complex contract expressions:")
    
    @require("data must be non-empty list", lambda args: isinstance(args.data, list) and len(args.data) > 0)
    @require("all elements must be numbers", lambda args: all(isinstance(x, (int, float)) for x in args.data))
    @ensure("result must be dictionary", lambda args, result: isinstance(result, dict))
    @ensure("result must contain mean and count", lambda args, result: 'mean' in result and 'count' in result)
    def calculate_statistics(data):
        """Calculate basic statistics with comprehensive contracts."""
        mean = sum(data) / len(data)
        return {'mean': mean, 'count': len(data), 'min': min(data), 'max': max(data)}
    
    @require("n must be positive", lambda args: args.n > 0)
    @require("n must be integer", lambda args: isinstance(args.n, int))
    @ensure("result length must equal n", lambda args, result: len(result) == args.n)
    @ensure("all fibonacci numbers are non-negative", lambda args, result: all(x >= 0 for x in result))
    def fibonacci_sequence(n):
        """Generate fibonacci sequence with contracts."""
        if n == 1:
            return [0]
        elif n == 2:
            return [0, 1]
        
        sequence = [0, 1]
        for i in range(2, n):
            sequence.append(sequence[i-1] + sequence[i-2])
        return sequence
    
    @require("base must be positive and not 1", lambda args: args.base > 0 and args.base != 1)
    @require("x must be positive", lambda args: args.x > 0)
    @ensure("result must be numeric", lambda args, result: isinstance(result, (int, float)))
    def logarithm(x, base=10):
        """Calculate logarithm with validation."""
        return math.log(x) / math.log(base)
    
    # Test complex contracts
    print("    Testing complex contract expressions:")
    
    test_cases = [
        ("calculate_statistics([1, 2, 3, 4, 5])", lambda: calculate_statistics([1, 2, 3, 4, 5])),
        ("fibonacci_sequence(8)", lambda: fibonacci_sequence(8)),
        ("logarithm(100, 10)", lambda: logarithm(100, 10))
    ]
    
    for test_name, test_func in test_cases:
        try:
            result = test_func()
            print(f"      {test_name} = {result}")
        except Exception as e:
            print(f"      {test_name} failed: {type(e).__name__}")
            raise
    
    # 2. Error handling and debugging
    print("  2. Contract debugging:")
    
    # Test contract violations for debugging
    # (call, the precondition message it must violate)
    error_cases = [
        ("calculate_statistics([])", lambda: calculate_statistics([]), "data must be non-empty list"),
        ("fibonacci_sequence(-1)", lambda: fibonacci_sequence(-1), "n must be positive"),
        ("logarithm(-5, 10)", lambda: logarithm(-5, 10), "x must be positive")
    ]
    
    print("    Testing contract violations:")
    for test_name, test_func, expected_message in error_cases:
        try:
            result = test_func()
        except PreconditionError as e:
            assert str(e) == expected_message, (test_name, str(e))
            print(f"      {test_name} correctly failed: {type(e).__name__}")
        else:
            raise AssertionError(f"{test_name} = {result}: expected PreconditionError({expected_message!r})")
    
    # 3. Best practices
    print("  3. Contract best practices:")
    
    best_practices = [
        "Use meaningful error messages",
        "Keep contracts simple and readable",
        "Test both valid and invalid inputs",
        "Use contracts for input validation",
        "Document assumptions explicitly"
    ]
    
    for practice in best_practices:
        print(f"      • {practice}")
    
    print("✓ Advanced contract patterns and debugging completed")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dpcontracts example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Integration Patterns and Production Considerations

In [ ]:
import dpcontracts
from dpcontracts import require, ensure

print("Example 5: Integration Patterns and Production Considerations")

try:
    print("✓ Integration patterns and production considerations:")
    
    # 1. Package information
    print("  1. Package information:")
    print(f"    dpcontracts module imported successfully")
    print(f"    Module type: {type(dpcontracts).__name__}")
    
    attrs = [attr for attr in dir(dpcontracts) if not attr.startswith('_')]
    print(f"    Public attributes: {len(attrs)}")
    print(f"    Key components: require, ensure, invariant")
    
    # 2. Simple integration example
    print("  2. Simple integration example:")
    
    @require("balance must be non-negative", lambda args: args.balance >= 0)
    @ensure("returned balance must be non-negative", lambda args, result: result >= 0)
    def simple_deposit(balance, amount):
        """Simple deposit function with contracts."""
        if amount <= 0:
            return balance
        return balance + amount
    
    @require("insufficient funds", lambda args: args.balance >= args.amount)
    @require("withdrawal amount must be positive", lambda args: args.amount > 0)
    @ensure("balance updated correctly", lambda args, result: result == args.balance - args.amount)
    def simple_withdraw(balance, amount):
        """Simple withdrawal function with contracts."""
        return balance - amount
    
    # Test integration examples
    print("    Testing integration examples:")
    
    try:
        result = simple_deposit(100, 50)
        print(f"      simple_deposit(100, 50) = {result}")
        
        result = simple_withdraw(150, 30)
        print(f"      simple_withdraw(150, 30) = {result}")
        
    except Exception as e:
        print(f"      Integration test failed: {type(e).__name__}")
        raise
    
    # 3. Production considerations
    print("  3. Production considerations:")
    
    considerations = [
        "Contracts add runtime overhead",
        "Useful for development and testing",
        "Consider disabling in production for performance",
        "Provide clear error messages for debugging",
        "Help catch bugs early in development cycle"
    ]
    
    for consideration in considerations:
        print(f"      • {consideration}")
    
    # 4. Contract patterns
    print("  4. Common contract patterns:")
    
    patterns = [
        "Input validation with preconditions",
        "Output validation with postconditions", 
        "State consistency with invariants",
        "Error handling with meaningful messages",
        "Documentation through executable specifications"
    ]
    
    for pattern in patterns:
        print(f"      • {pattern}")
    
    print("✓ Integration patterns completed successfully")
    
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"dpcontracts example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")